# Galilean IMU preintegration: a four-backend NEES comparison

This notebook compares GTSAM's Manifold, Tangent, Lie-group, and Galilean IMU preintegration backends under identical measurements and noise samples. The stress test uses simultaneous body-frame acceleration and rotation with a zero-order-held IMU signal. That is the regime in which the Galilean exponential integrates the coupled rotation, velocity, position, and time dynamics exactly.

The word *better* has two testable meanings here: smaller deterministic endpoint error at a fixed sample period, and statistical consistency measured by 9D Normalized Estimation Error Squared (NEES). This experiment does not claim universal dominance, lower runtime, or an advantage when the assumed held-input model is a poor description of the sensor signal. See the companion [`GalileanImuFactor` guide](GalileanImuFactor.ipynb) for the complete left-invariant derivation.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/navigation/doc/GalileanImuFactorNEES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from scipy.stats import chi2

import gtsam

np.set_printoptions(precision=4, suppress=True)
pio.renderers.default = "notebook_connected"

## 1. What NEES measures

For the factor residual $e\in\mathbb R^9$ and its predicted covariance $P$,

$$
\operatorname{NEES}=e^\mathsf{T}P^{-1}e.
$$

A consistent model has expected NEES equal to the residual dimension, nine. For $N$ independent trials, the exact 95% acceptance interval for the average NEES is

$$
\left[\frac{\chi^2_{0.025,9N}}{N},\frac{\chi^2_{0.975,9N}}{N}\right].
$$

Every backend below receives the same continuous-time accelerometer and gyroscope noise densities, the same sampled noise realization in each trial, the same zero bias, and the same exact endpoint. Gravity and integration noise are set to zero to isolate the preintegration discretization and its sensor-noise covariance. The evaluated error is `state_j.localCoordinates(predicted_state_j)`, exactly the existing GTSAM IMU factor residual chart.

In [3]:
DURATION = 1.0
ACCELERATION = np.array([2.0, -1.0, 0.5])
ANGULAR_VELOCITY = np.array([1.2, -0.8, 2.0])
ACCELEROMETER_SIGMAS = np.array([0.03, 0.04, 0.05])
GYROSCOPE_SIGMAS = np.array([0.01, 0.015, 0.02])
BIAS = gtsam.imuBias.ConstantBias()
STATE_I = gtsam.NavState()

BACKENDS = {
    "Manifold": gtsam.PreintegratedImuMeasurementsManifold,
    "Tangent": gtsam.PreintegratedImuMeasurementsTangent,
    "Lie group": gtsam.PreintegratedImuMeasurementsLieGroup,
    "Galilean": gtsam.PreintegratedImuMeasurementsG,
}


def make_params():
    params = gtsam.PreintegrationParams(np.zeros(3))
    params.setAccelerometerCovariance(
        np.diag(ACCELEROMETER_SIGMAS**2)
    )
    params.setGyroscopeCovariance(np.diag(GYROSCOPE_SIGMAS**2))
    params.setIntegrationCovariance(np.zeros((3, 3)))
    return params


PARAMS = make_params()

## 2. Independent closed-form endpoint

With constant body acceleration $a$, angular rate $\omega$, and duration $T$, the continuous held-input solution is the Galilean exponential

$$
\Upsilon(T)=\operatorname{Exp}_{\mathrm{Gal}(3)}(T(\omega,a,0,1)).
$$

This is the analytical solution of the continuous dynamics, not a fine-step numerical reference. We remove deterministic time and map $(R,v,p,T)$ into the `NavState` ordering $(R,p,v)`.

In [4]:
def exact_held_input_state(acceleration, angular_velocity, duration):
    tangent = np.zeros(10)
    tangent[:3] = angular_velocity * duration
    tangent[3:6] = acceleration * duration
    tangent[9] = duration
    delta = gtsam.Gal3.Expmap(tangent)
    return gtsam.NavState(
        delta.rotation(), delta.translation(), delta.velocity()
    )


TRUTH = exact_held_input_state(
    ACCELERATION, ANGULAR_VELOCITY, DURATION
)


def integrate(backend_type, accelerations, angular_velocities, dt):
    pim = backend_type(PARAMS, BIAS)
    for acceleration, angular_velocity in zip(
        accelerations, angular_velocities
    ):
        pim.integrateMeasurement(acceleration, angular_velocity, dt)
    return pim

## 3. Deterministic discretization error

First remove sensor noise and vary only the sample period. The three established backends use the same piecewise update for this trajectory and therefore overlap. Their error decreases linearly as the timestep shrinks. Galilean composition remains at floating-point precision because each held-input interval uses the exact coupled exponential.

In [5]:
SAMPLE_PERIODS = np.array([0.1, 0.05, 0.025, 0.0125])
deterministic = {
    name: {"position": [], "velocity": []}
    for name in BACKENDS
}

for dt in SAMPLE_PERIODS:
    steps = round(DURATION / dt)
    accelerations = np.tile(ACCELERATION, (steps, 1))
    angular_velocities = np.tile(ANGULAR_VELOCITY, (steps, 1))
    for name, backend_type in BACKENDS.items():
        pim = integrate(
            backend_type, accelerations, angular_velocities, dt
        )
        error = np.asarray(
            TRUTH.localCoordinates(pim.predict(STATE_I, BIAS))
        )
        deterministic[name]["position"].append(
            np.linalg.norm(error[3:6])
        )
        deterministic[name]["velocity"].append(
            np.linalg.norm(error[6:9])
        )

assert max(deterministic["Galilean"]["position"]) < 2e-12
assert max(deterministic["Galilean"]["velocity"]) < 4e-12
for name in ("Manifold", "Tangent", "Lie group"):
    assert deterministic[name]["position"][1] > 0.03
    assert deterministic[name]["velocity"][1] > 0.07

In [11]:
fig = go.Figure()
line_styles = {
    "Manifold": ("#777777", "solid"),
    "Tangent": ("#999999", "dash"),
    "Lie group": ("#bbbbbb", "dot"),
    "Galilean": ("#14866d", "solid"),
}
for name in BACKENDS:
    color, dash = line_styles[name]
    fig.add_scatter(
        x=SAMPLE_PERIODS,
        y=deterministic[name]["velocity"],
        mode="lines+markers",
        name=name,
        line=dict(color=color, dash=dash),
    )
fig.update_xaxes(type="log", title="IMU sample period (s)")
fig.update_yaxes(type="log", title="Velocity error norm (m/s)")
fig.update_layout(
    title="Held-input discretization error",
    template="plotly_white",
    legend_title_text="Backend",
)
fig.show()

## 4. Paired Monte Carlo NEES experiment

We now use a 20 Hz IMU for one second and add anisotropic white sensor noise. Continuous-time noise density $\sigma$ becomes sampled rate noise $\sigma/\sqrt{\Delta t}$. Each trial generates one noise sequence and feeds that identical sequence to all four backends, making this a paired comparison. Each backend supplies its own propagated `residualCovariance()`.

In [7]:
DT = 0.05
STEPS = round(DURATION / DT)
TRIALS = 3_000
SEED = 2231

rng = np.random.default_rng(SEED)
errors = {name: np.empty((TRIALS, 9)) for name in BACKENDS}
nees = {name: np.empty(TRIALS) for name in BACKENDS}

for trial in range(TRIALS):
    accelerometer_noise = rng.normal(size=(STEPS, 3)) * (
        ACCELEROMETER_SIGMAS / np.sqrt(DT)
    )
    gyroscope_noise = rng.normal(size=(STEPS, 3)) * (
        GYROSCOPE_SIGMAS / np.sqrt(DT)
    )
    accelerations = ACCELERATION + accelerometer_noise
    angular_velocities = ANGULAR_VELOCITY + gyroscope_noise

    for name, backend_type in BACKENDS.items():
        pim = integrate(
            backend_type, accelerations, angular_velocities, DT
        )
        error = np.asarray(
            TRUTH.localCoordinates(pim.predict(STATE_I, BIAS))
        )
        covariance = np.asarray(pim.residualCovariance())
        errors[name][trial] = error
        nees[name][trial] = error @ np.linalg.solve(covariance, error)

## 5. Results

The expected band below is the exact chi-square interval for the average of 3,000 independent 9D NEES samples. The error bars on each point are empirical 95% confidence intervals for that backend's sampled mean. Position and velocity RMS values are norms within their three-dimensional blocks, so they retain physical units.

In [8]:
DIMENSION = 9
expected_interval = chi2.ppf(
    [0.025, 0.975], TRIALS * DIMENSION
) / TRIALS

summary = {}
for name in BACKENDS:
    backend_errors = errors[name]
    values = nees[name]
    mean = values.mean()
    mean_half_width = 1.96 * values.std(ddof=1) / np.sqrt(TRIALS)
    summary[name] = {
        "mean_nees": mean,
        "mean_half_width": mean_half_width,
        "rotation_rmse": np.sqrt(
            np.mean(np.sum(backend_errors[:, :3] ** 2, axis=1))
        ),
        "position_rmse": np.sqrt(
            np.mean(np.sum(backend_errors[:, 3:6] ** 2, axis=1))
        ),
        "velocity_rmse": np.sqrt(
            np.mean(np.sum(backend_errors[:, 6:9] ** 2, axis=1))
        ),
        "mean_error_norm": np.linalg.norm(backend_errors.mean(axis=0)),
    }

rows = [
    "| Backend | Mean NEES | Position RMS (m) | Velocity RMS (m/s) | Mean error norm |",
    "|---|---:|---:|---:|---:|",
]
for name, values in summary.items():
    rows.append(
        f"| {name} | {values['mean_nees']:.3f} | "
        f"{values['position_rmse']:.4f} | "
        f"{values['velocity_rmse']:.4f} | "
        f"{values['mean_error_norm']:.4f} |"
    )
display(Markdown("\n".join(rows)))
print(
    "Expected 95% interval for mean NEES: "
    f"[{expected_interval[0]:.3f}, {expected_interval[1]:.3f}]"
)

| Backend | Mean NEES | Position RMS (m) | Velocity RMS (m/s) | Mean error norm |
|---|---:|---:|---:|---:|
| Manifold | 14.506 | 0.0578 | 0.1049 | 0.0818 |
| Tangent | 14.511 | 0.0578 | 0.1049 | 0.0818 |
| Lie group | 14.506 | 0.0578 | 0.1049 | 0.0818 |
| Galilean | 8.958 | 0.0427 | 0.0767 | 0.0011 |

Expected 95% interval for mean NEES: [8.849, 9.152]


In [9]:
# Executable statistical claims for this deterministic experiment.
galilean_nees = summary["Galilean"]["mean_nees"]
assert expected_interval[0] <= galilean_nees <= expected_interval[1]

for name in ("Manifold", "Tangent", "Lie group"):
    assert summary[name]["mean_nees"] > expected_interval[1]
    assert (
        summary["Galilean"]["position_rmse"]
        < summary[name]["position_rmse"]
    )
    assert (
        summary["Galilean"]["velocity_rmse"]
        < summary[name]["velocity_rmse"]
    )

In [10]:
names = list(BACKENDS)
means = [summary[name]["mean_nees"] for name in names]
half_widths = [
    summary[name]["mean_half_width"] for name in names
]
colors = ["#888888", "#999999", "#aaaaaa", "#14866d"]

fig = go.Figure()
fig.add_hrect(
    y0=expected_interval[0],
    y1=expected_interval[1],
    fillcolor="#14866d",
    opacity=0.14,
    line_width=0,
    annotation_text="95% consistency band",
    annotation_position="top left",
)
fig.add_scatter(
    x=names,
    y=means,
    mode="markers",
    marker=dict(size=11, color=colors),
    error_y=dict(type="data", array=half_widths, visible=True),
    hovertemplate="%{x}: mean NEES %{y:.3f}<extra></extra>",
)
fig.add_hline(y=DIMENSION, line_dash="dash", line_color="#333333")
fig.update_layout(
    title="Average 9D NEES under identical high-dynamic IMU samples",
    xaxis_title="Preintegration backend",
    yaxis_title="Mean NEES",
    yaxis_range=[0, max(means) + 1.5],
    template="plotly_white",
    showlegend=False,
)
fig.show()

## 6. Interpretation and limits

With the fixed seed, Galilean preintegration has mean NEES close to nine and inside the exact 95% consistency interval. Manifold, Tangent, and Lie-group preintegration all lie well above the interval. Their propagated covariances describe sensor noise, but their deterministic position/velocity discretization error is not represented in that covariance, so the residual is overconfident.

The endpoint table also separates consistency from accuracy: Galilean preintegration reduces both position and velocity RMS error, while all four backends have essentially the same rotation RMS error. The timestep sweep identifies the cause. The three established variants converge as the sampling interval shrinks, whereas the Galilean group law and exponential integrate the coupled held input exactly at every tested interval.

This result is deliberately scoped. At very high IMU rates the standard discretization error becomes negligible; with time-varying input inside a sample, all zero-order-hold methods inherit model error; and this notebook does not compare runtime, bias random walks, sensor-pose corrections, or full graph optimization. It shows that for simultaneous rotation and acceleration at a finite sampling rate, the left-invariant Galilean formulation provides the most accurate mean and the only statistically consistent covariance among the four tested GTSAM backends.